# Text-to-SQL QLoRA training (Kaggle, backup environment)

Same `src/train.py` as Colab — only the checkpoint-mount mechanism differs. Mounts the *same* Google Drive `checkpoints/` folder Colab writes to via rclone, so `get_last_checkpoint()` finds Colab-written checkpoints directly (e.g. `ablation_r16/checkpoint-50`) with no manual transfer.

**One-time setup required first:** a Kaggle Secret named `RCLONE_CONF` containing your rclone Google Drive remote config. See the project conversation/README for the step-by-step OAuth + Kaggle Secret walkthrough. Make sure the secret is toggled on for this notebook (Add-ons → Secrets).

**Differences from `train_colab.ipynb` worth knowing:**
- No Drive auth popup — token-based via the Kaggle Secret instead.
- Working directory defaults to `/kaggle/working`, not `/content`.
- Kaggle enforces a hard 30 GPU-hr/week quota with no idle-disconnect grace period the way Colab does — start sessions deliberately. If a session ends, re-running this notebook resumes from the latest checkpoint automatically; there's no "reconnect to the same session," only "rerun and resume."
- Enable the GPU: Notebook settings (right sidebar) → Accelerator → **GPU T4 x2**. `train.py` only uses one GPU (no multi-GPU logic), so the second T4 is idle — fine for our purposes, just don't expect a speedup from it.

## Cell 1 — rclone setup + Drive mount

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, time

conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(conf_dir, exist_ok=True)
with open(os.path.join(conf_dir, "rclone.conf"), "w") as f:
    f.write(UserSecretsClient().get_secret("RCLONE_CONF"))

!curl https://rclone.org/install.sh | sudo bash

MOUNT_POINT = "/kaggle/working/drive_checkpoints"
os.makedirs(MOUNT_POINT, exist_ok=True)
os.environ['MOUNT_POINT'] = MOUNT_POINT

!nohup rclone mount gdrive:text-to-sql-qlora/checkpoints "$MOUNT_POINT" --vfs-cache-mode writes --daemon
time.sleep(5)  # give the mount a moment to come up before we rely on it

print("Mount contents (should show ablation_r8/, ablation_r16/, etc. if Colab already wrote checkpoints there):")
!ls "$MOUNT_POINT"

## Cell 2 — clone + install

In [ ]:
import os
REPO_DIR = '/kaggle/working/text-to-sql-qlora'  # explicit dir name — avoids the leading '-' in the GitHub repo name, which breaks %cd's flag parser
os.environ['REPO_DIR'] = REPO_DIR

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone https://github.com/Rick-Roy-JC/-text-to-sql-qlora.git "$REPO_DIR"
else:
    print(f"{REPO_DIR} already cloned — pulling latest instead.")
    !git -C "$REPO_DIR" pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

## Cell 3 — resume r=16 from the Colab-written checkpoint

`--output-root` points at the rclone mount, which is the same Drive folder Colab used — `get_last_checkpoint()` will find `checkpoint-50` (or later, if Colab progressed further before you switch over) automatically. The scaler-guard fix in `train.py` handles the `paged_adamw_8bit` + fp16 resume issue automatically now.

In [ ]:
%cd {REPO_DIR}
!python src/train.py --regime ablation --lora-rank 16 --output-root "$MOUNT_POINT"

## Cell 4 — run r=32 (fresh)

In [ ]:
%cd {REPO_DIR}
!python src/train.py --regime ablation --lora-rank 32 --output-root "$MOUNT_POINT"